# **ML Assignment 2**


---

In [ ]:
from math import log, inf
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import seaborn as sns
from plotly.subplots import make_subplots
import math
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler, TargetEncoder, PolynomialFeatures,MinMaxScaler,OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split,cross_val_score,StratifiedKFold,KFold
from sklearn.linear_model import LinearRegression, Ridge, Lasso,LogisticRegression
from sklearn.naive_bayes import GaussianNB ,BernoulliNB,ComplementNB
from sklearn.compose import ColumnTransformer
from sklearn.metrics import r2_score,confusion_matrix,precision_score,f1_score,recall_score,ConfusionMatrixDisplay,accuracy_score,classification_report




df = pd.read_csv('Fifa.csv')

# EDA

In [ ]:
df.info()

In [ ]:
df

In [ ]:
df.describe()

In [ ]:
df[df['Value Per M$'] >= 100]

In [ ]:
df[df['Value Per M$'] == 0]

330 rows with Value Per M$ is zero

In [ ]:
df.isna().sum()

Data contains no missing values.

In [ ]:
df.nunique()

In [ ]:
for col in df.columns:
    print(df[col].value_counts())

In [ ]:
plt.figure(figsize=(10, 6))
sns.histplot(df['Value Per M$'], kde=True, color='purple', bins=40)
plt.title('Distribution of Values')
plt.show()

We can see the distribution of the column "Value Per M$" is right skewed, so we apply log transformation to make it more normal distributed, and this will help the model to learn better

In [ ]:
numerical_features = df.select_dtypes(include=np.number).columns
categorical_features = df.drop(columns=['Name']).select_dtypes(exclude=np.number).columns

sns.heatmap(df[numerical_features].corr(), annot=True, cmap='coolwarm')

"Future Potential" and "Overall_Rating" are the most correlated features to the target "Value Per M$"

In [ ]:
df['Overall_Rating'].groupby(df['Position']).mean().plot(kind='bar')

["RF: Right Forward",  "SW: Sweeper"] are the top positions with high overall rating

In [ ]:
overall_fig = px.bar(df['Overall_Rating'].value_counts(), title='Overall Rating by Position')

In [ ]:

num_cols = 3
num_rows = math.ceil(len(numerical_features) / num_cols)

fig = make_subplots(
    rows=num_rows,
    cols=num_cols,
    subplot_titles=list(numerical_features)
)

for i, feature in enumerate(numerical_features):
    row = (i // num_cols) + 1
    col = (i % num_cols) + 1

    fig.add_trace(
        go.Histogram(x=df[feature], name=feature),
        row=row,
        col=col
    )

fig.update_layout(
    height=400 * num_rows,
    width=900,
    title_text="Distribution of Numerical Features",
    showlegend=False
)

fig.show()

We can see that the "Age" distribution is a little bit right-skewed, but this is normal because the majority of Football players are young

In [ ]:
most_expensive_teams = df[['Team', 'Value Per M$']].groupby('Team').mean().sort_values(by='Value Per M$', ascending=False).head(10)

In [ ]:
px.bar(most_expensive_teams, x=most_expensive_teams.index, y='Value Per M$', title='Top 10 Most Expensive Teams')

In [ ]:
most_expensive_positions = df[['Position', 'Value Per M$']].groupby('Position').mean().sort_values(by='Value Per M$', ascending=False).head(10)

In [ ]:
px.bar(most_expensive_positions, x=most_expensive_positions.index, y='Value Per M$', title='Top 10 Most Expensive Positions')

"CF: Centre Forward" is the most expensive position, after it the "LW: Left Winger"

In [ ]:
for column in numerical_features:
    plt.figure(figsize=(10, 6))
    plt.boxplot(df[column].dropna(), vert=False)
    plt.title(f'Box Plot of {column}')
    plt.xlabel(column)
    plt.grid(axis='x', alpha=0.75)
    plt.show()

In [ ]:
for column in numerical_features:
    q1 = df[column].quantile(0.25)
    q3 = df[column].quantile(0.75)
    IQR = q3 - q1
    lower_bound = q1 - 1.5 * IQR
    upper_bound = q3 + 1.5 * IQR
    print(f"Lower Bound: {lower_bound}, Upper Bound: {upper_bound}")
    outliers = df[(df[column] < lower_bound) | (df[column] > upper_bound)]
    print(f"Number of outliers in '{column}': {outliers.shape[0]}")
    print('='*20)



# Data Preprocessing

Split data into train and test sets, and save the pipelines for data preprocessing to apply the same transformations on the test set.

In [ ]:
trainX,testX,trainY,testY = train_test_split(df.drop(columns=['Value Per M$']), df['Value Per M$'], test_size=0.2, random_state=42)

In [ ]:
numerical_features = df.drop(columns=['Value Per M$']).select_dtypes(include=np.number).columns
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_features),
        ('cat', TargetEncoder(), categorical_features)
    ])

pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),])
pipeline.set_output(transform="pandas")
trainX_before_pipeline = trainX.copy()
testX_before_pipeline = testX.copy()
trainX = pipeline.fit_transform(trainX, trainY)
testX = pipeline.transform(testX)

Apply log transformation to the target variable `"Value Per M$"`

In [ ]:
trainY = trainY.apply(lambda x: log(x) if x > 0 else 0)
testY = testY.apply(lambda x: log(x) if x > 0 else 0)

In [ ]:
plt.figure(figsize=(10, 6))
sns.histplot(trainY, kde=True, color='purple', bins=40)
plt.title('Distribution of Values')
plt.show()
plt.figure(figsize=(10, 6))
plt.boxplot(trainY, vert=False)
plt.title(f'Box Plot of Value Per M$')
plt.xlabel('Value Per M$')
plt.grid(axis='x', alpha=0.75)
plt.show()

#Task 3 : Create Classification Target

In [ ]:
# Calculate quartiles for 'Overall_Rating' on the training data

Q1 = trainX['num__Overall_Rating'].quantile(0.75)
Q2 = trainX['num__Overall_Rating'].quantile(0.82)
Q3 = trainX['num__Overall_Rating'].quantile(0.95)


# Define a function to categorize players into tiers
def categorize_rating(rating, Q1, Q2, Q3):
    if rating <= Q1:
        return 'Low'
    elif rating <= Q2:
        return 'Mid'
    elif rating <= Q3:
        return 'High'
    else:
        return 'Elite'

# Apply the categorization to both training and test dataframes
trainX['Rating_Tier'] = trainX['num__Overall_Rating'].apply(lambda x: categorize_rating(x, Q1, Q2, Q3))
testX['Rating_Tier'] = testX['num__Overall_Rating'].apply(lambda x: categorize_rating(x, Q1, Q2, Q3))

In [ ]:
#Count of players in each tier in the training data
tier_counts = trainX['Rating_Tier'].value_counts()
print("Distribution of players across Rating Tiers:")
print(tier_counts)

In [ ]:
# Plotting the class distribution
plt.figure(figsize=(10, 6))
sns.barplot(x=tier_counts.index, y=tier_counts.values)
plt.title('Distribution of Player Performance Tiers')
plt.xlabel('Performance Tier')
plt.ylabel('Number of Players')
plt.show()

The class distribution appears relatively balanced.

# Task 4 : Polynomial Regression

In [ ]:
trainX['Rating_Tier'] = trainX['Rating_Tier'].map({'Low' : 0,'Mid' : 1,'High' : 2,'Elite' : 3})
testX['Rating_Tier'] = testX['Rating_Tier'].map({'Low' : 0,'Mid' : 1,'High' : 2,'Elite' : 3})

- Normal Linear Regression

In [ ]:
normal_linear_regression = LinearRegression()
normal_linear_regression.fit(trainX, trainY)
normal_linear_regression_train_pred = normal_linear_regression.predict(trainX)
normal_linear_regression_test_pred = normal_linear_regression.predict(testX)
normal_linear_regression_train_r2 = r2_score(trainY, normal_linear_regression_train_pred)
normal_linear_regression_test_r2 = r2_score(testY, normal_linear_regression_test_pred)
normal_linear_regression_mae = np.mean(np.abs(normal_linear_regression_test_pred - testY))
normal_linear_regression_mse = np.mean((normal_linear_regression_test_pred - testY) ** 2)
normal_linear_regression_rmse = np.sqrt(normal_linear_regression_mse)

print(f"Normal Linear Regression Train R^2 Score: {(normal_linear_regression_train_r2*100).__format__('.4f')}")
print(f"Normal Linear Regression Test R^2 Score: {(normal_linear_regression_test_r2*100).__format__('.4f')}")
print(f"Normal Linear Regression MAE: {(normal_linear_regression_mae).__format__('.4f')}")
print(f"Normal Linear Regression MSE: {(normal_linear_regression_mse).__format__('.4f')}")
print(f"Normal Linear Regression RMSE: {(normal_linear_regression_rmse).__format__('.4f')}")



- Normal Linear Regression with Polynomial Features

In [ ]:
test_scores = []
train_scores = []
for degree in range(1, 5):
    poly_features = PolynomialFeatures(degree=degree)
    trainX_poly = poly_features.fit_transform(trainX.drop(columns=['cat__Country','cat__Position','cat__Team','Rating_Tier']))
    testX_poly = poly_features.transform(testX.drop(columns=['cat__Country','cat__Position','cat__Team','Rating_Tier']))

    model = LinearRegression()
    model.fit(trainX_poly, trainY)

    trainY_pred = model.predict(trainX_poly)
    testY_pred = model.predict(testX_poly)
    train_Accuracy = r2_score(trainY, trainY_pred)
    test_Accuracy = r2_score(testY, testY_pred)
    test_scores.append(model.score(testX_poly, testY))
    train_scores.append(model.score(trainX_poly, trainY))
    print(f"Polynomial Regression (Degree {degree}) Train R^2 Score: {(train_Accuracy*100).__format__('.4f')}")
    print(f"Polynomial Regression (Degree {degree}) Test R^2 Score: {(test_Accuracy*100).__format__('.4f')}")

In [ ]:
fig = go.Figure(data=[
    go.Bar(name='Train Accuracy', x=[1,2,3,4],y=train_scores),
    go.Bar(name='Test Accuracy', x=[1,2,3,4],y=test_scores),
])
fig.update_layout(barmode='group')
fig.show()

Best degree is 4, because it has the highest test R^2 score

In [ ]:
poly_features = PolynomialFeatures(degree=4)
trainX_poly = poly_features.fit_transform(trainX.drop(columns=['cat__Country', 'cat__Position', 'cat__Team', 'Rating_Tier']),)
testX_poly = poly_features.transform(testX.drop(columns=['cat__Country', 'cat__Position', 'cat__Team', 'Rating_Tier']))
column_names = poly_features.get_feature_names_out()


- Regularized Linear Regression with Polynomial Features (Ridge Regression)

In [ ]:
test_rmse_ridge = []
train_rmse_ridge = []
best_alpha_ridge = 0
best_rmse_ridge = float("inf")
for alpha in np.logspace(-2,0,20):
    ridge = Ridge(alpha=alpha)
    ridge.fit(trainX_poly, trainY)
    ridge_test_pred = ridge.predict(testX_poly)
    ridge_train_pred = ridge.predict(trainX_poly)

    #Calculating RMSE for Train Data
    ridge_train_linear_regression_mse = np.mean((ridge_train_pred - trainY) ** 2)
    ridge_train_linear_regression_rmse =np.sqrt(ridge_train_linear_regression_mse)
    train_rmse_ridge.append(ridge_train_linear_regression_rmse)

    #Calculating RMSE for Test Data
    ridge_test_linear_regression_mse = np.mean((ridge_test_pred - testY) ** 2)
    ridge_test_linear_regression_rmse =np.sqrt(ridge_test_linear_regression_mse)
    test_rmse_ridge.append(ridge_test_linear_regression_rmse)


    if ridge_test_linear_regression_rmse < best_rmse_ridge:
        best_rmse_ridge = ridge_test_linear_regression_rmse
        best_alpha_ridge = alpha
    print(f"Polynomial Regression Train RMSE: {ridge_train_linear_regression_rmse.__format__('.4f')}")
    print(f"Polynomial Regression Test RMSE: {ridge_test_linear_regression_rmse.__format__('.4f')}")

print(f"Best Polynomial Regression Test RMSE: {best_rmse_ridge.__format__('.4f')} with alpha: {best_alpha_ridge}")


In [ ]:
fig = go.Figure(data=[
    go.Bar(name='Train RMSE', x=np.logspace(-2,0,20),y=train_rmse_ridge,),
    go.Bar(name='Test RMSE', x=np.logspace(-2,0,20),y=test_rmse_ridge),
])
fig.update_layout(barmode='group')
fig.show()

- Regularized Linear Regression with Polynomial Features (Lasso Regression)

In [ ]:
test_rmse_lasso = []
train_rmse_lasso = []
best_alpha_lasso = 0
best_rmse_lasso = float("inf")
best_lasso = Lasso()
for alpha in np.logspace(-2,0,20):
    lasso = Lasso(alpha=alpha)

    lasso.fit(trainX_poly, trainY)
    print(lasso.coef_)
    lasso_test_pred = lasso.predict(testX_poly)
    lasso_train_pred = lasso.predict(trainX_poly)

    #Calculating RMSE for Train Data
    lasso_train_linear_regression_mse = np.mean((lasso_train_pred - trainY) ** 2)
    lasso_train_linear_regression_rmse =np.sqrt(lasso_train_linear_regression_mse)
    train_rmse_lasso.append(lasso_train_linear_regression_rmse)

    #Calculating RMSE for Test Data
    lasso_test_linear_regression_mse = np.mean((lasso_test_pred - testY) ** 2)
    lasso_test_linear_regression_rmse =np.sqrt(lasso_test_linear_regression_mse)
    test_rmse_lasso.append(lasso_test_linear_regression_rmse)


    if lasso_test_linear_regression_rmse < best_rmse_lasso:
        best_rmse_lasso = lasso_test_linear_regression_rmse
        best_alpha_lasso = alpha
        best_lasso = lasso
    print(f"Polynomial Regression Train RMSE: {lasso_train_linear_regression_rmse.__format__('.4f')}")
    print(f"Polynomial Regression Test RMSE: {lasso_test_linear_regression_rmse.__format__('.4f')}")
print(f"Best Polynomial Regression Test RMSE: {best_rmse_lasso.__format__('.4f')} with alpha: {best_alpha_lasso}")


In [ ]:
fig = go.Figure(data=[
    go.Bar(name='Train RMSE', x=np.logspace(-2,0,20),y=train_rmse_lasso),
    go.Bar(name='Test RMSE', x=np.logspace(-2,0,20),y=test_rmse_lasso),
])
fig.update_layout(barmode='group')
fig.show()

In [ ]:
important_features = pd.Series(best_lasso.coef_,index=column_names)

selected_features = important_features[important_features != 0]

print(selected_features)

# Logistic Regression

In [ ]:
logistic_trainX = trainX.drop(columns=['Rating_Tier','num__Overall_Rating'])
logistic_testX = testX.drop(columns=['Rating_Tier','num__Overall_Rating'])
logistic_trainY = trainX['Rating_Tier']
logistic_testY = testX['Rating_Tier']

logistic_trainY = logistic_trainY.map({0 : 'Low', 1 : 'Mid', 2 : 'High', 3 : 'Elite'})
logistic_testY = logistic_testY.map({0 : 'Low', 1 : 'Mid', 2 : 'High', 3 : 'Elite'})

logistic_regression = LogisticRegression()

logistic_regression.fit(logistic_trainX, logistic_trainY)

logistic_train_pred = logistic_regression.predict(logistic_trainX)
logistic_test_pred = logistic_regression.predict(logistic_testX)

logistic_train_accuracy = logistic_regression.score(logistic_trainX, logistic_trainY)
logistic_test_accuracy = logistic_regression.score(logistic_testX, logistic_testY)

logistic_train_precision = precision_score(logistic_trainY, logistic_train_pred, average='weighted')
logistic_test_precision = precision_score(logistic_testY, logistic_test_pred, average='weighted')

logistic_train_recall = recall_score(logistic_trainY, logistic_train_pred, average='weighted')
logistic_test_recall = recall_score(logistic_testY, logistic_test_pred, average='weighted')

logistic_train_f1 = f1_score(logistic_trainY, logistic_train_pred, average='weighted')
logistic_test_f1 = f1_score(logistic_testY, logistic_test_pred, average='weighted')

print(f"Logistic Regression Train Accuracy: {logistic_train_accuracy.__format__('.4f')}")
print(f"Logistic Regression Test Accuracy: {logistic_test_accuracy.__format__('.4f')}")

print(f"Logistic Regression Train Precision: {logistic_train_precision.__format__('.4f')}")
print(f"Logistic Regression Test Precision: {logistic_test_precision.__format__('.4f')}")

print(f"Logistic Regression Train Recall: {logistic_train_recall.__format__('.4f')}")
print(f"Logistic Regression Test Recall: {logistic_test_recall.__format__('.4f')}")

print(f"Logistic Regression Train F1 Score: {logistic_train_f1.__format__('.4f')}")
print(f"Logistic Regression Test F1 Score: {logistic_test_f1.__format__('.4f')}")

In [ ]:
cm = confusion_matrix(logistic_testY, logistic_test_pred, labels=['Low', 'Mid', 'High', 'Elite'])
# Plot the confusion matrix
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Low', 'Mid', 'High', 'Elite'], yticklabels=['Low', 'Mid', 'High', 'Elite'])
plt.title('Confusion Matrix for Logistic Regression')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.show()

In [ ]:
logistic_trainX = trainX.drop(columns=['Rating_Tier','num__Overall_Rating'])
logistic_testX = testX.drop(columns=['Rating_Tier','num__Overall_Rating'])
logistic_trainY = trainX['Rating_Tier']
logistic_testY = testX['Rating_Tier']

logistic_trainY = logistic_trainY.map({0 : 'Low', 1 : 'Mid', 2 : 'High', 3 : 'Elite'})
logistic_testY = logistic_testY.map({0 : 'Low', 1 : 'Mid', 2 : 'High', 3 : 'Elite'})

test_score_lasso = []
train_score_lasso = []
best_c = 0
best_accuracy = float('inf') * -1

for c in np.logspace(-3,3,20):
  logistic_regression = LogisticRegression(C=c,max_iter=10000)

  logistic_regression.fit(logistic_trainX, logistic_trainY)

  logistic_train_pred = logistic_regression.predict(logistic_trainX)
  logistic_test_pred = logistic_regression.predict(logistic_testX)

  logistic_train_accuracy = logistic_regression.score(logistic_trainX, logistic_trainY)
  logistic_test_accuracy = logistic_regression.score(logistic_testX, logistic_testY)

  train_score_lasso.append(logistic_train_accuracy)
  test_score_lasso.append(logistic_test_accuracy)

  if logistic_test_accuracy > best_accuracy:
    best_accuracy = logistic_test_accuracy
    best_c = c

  logistic_train_precision = precision_score(logistic_trainY, logistic_train_pred, average='weighted')
  logistic_test_precision = precision_score(logistic_testY, logistic_test_pred, average='weighted')

  logistic_train_recall = recall_score(logistic_trainY, logistic_train_pred, average='weighted')
  logistic_test_recall = recall_score(logistic_testY, logistic_test_pred, average='weighted')

  logistic_train_f1 = f1_score(logistic_trainY, logistic_train_pred, average='weighted')
  logistic_test_f1 = f1_score(logistic_testY, logistic_test_pred, average='weighted')

  print(f"Logistic Regression Train Accuracy: {logistic_train_accuracy.__format__('.4f')}")
  print(f"Logistic Regression Test Accuracy: {logistic_test_accuracy.__format__('.4f')}")

  print(f"Logistic Regression Train Precision: {logistic_train_precision.__format__('.4f')}")
  print(f"Logistic Regression Test Precision: {logistic_test_precision.__format__('.4f')}")

  print(f"Logistic Regression Train Recall: {logistic_train_recall.__format__('.4f')}")
  print(f"Logistic Regression Test Recall: {logistic_test_recall.__format__('.4f')}")

  print(f"Logistic Regression Train F1 Score: {logistic_train_f1.__format__('.4f')}")
  print(f"Logistic Regression Test F1 Score: {logistic_test_f1.__format__('.4f')}")

print(f"Best Logistic Regression Test Accuracy: {best_accuracy.__format__('.4f')} with C: {best_c}")

In [ ]:
fig = go.Figure(data=[
    go.Bar(name='Train Accuracy', x=np.logspace(-3,3,20),y=train_score_lasso),
    go.Bar(name='Test Accuracy', x=np.logspace(-3,3,20),y=test_score_lasso),
])
fig.update_layout(barmode='group')
fig.show()

Testing Best C on different solvers

In [ ]:
l1_logistic_regression = LogisticRegression(penalty='l1', solver='saga', C=best_c,max_iter=10000)
l1_logistic_regression.fit(logistic_trainX, logistic_trainY)
l1_logistic_train_pred = l1_logistic_regression.predict(logistic_trainX)
l1_logistic_test_pred = l1_logistic_regression.predict(logistic_testX)
l1_logistic_train_accuracy = l1_logistic_regression.score(logistic_trainX, logistic_trainY)
l1_logistic_test_accuracy = l1_logistic_regression.score(logistic_testX, logistic_testY)
print(f"Logistic Regression Train Accuracy: {l1_logistic_train_accuracy.__format__('.4f')}")
print(f"Logistic Regression Test Accuracy: {l1_logistic_test_accuracy.__format__('.4f')}")

In [ ]:
l2_logistic_regression = LogisticRegression(penalty='l2', solver='lbfgs', C=best_c,max_iter=10000)
l2_logistic_regression.fit(logistic_trainX, logistic_trainY)
l2_logistic_train_pred = l2_logistic_regression.predict(logistic_trainX)
l2_logistic_test_pred = l2_logistic_regression.predict(logistic_testX)
l2_logistic_train_accuracy = l2_logistic_regression.score(logistic_trainX, logistic_trainY)
l2_logistic_test_accuracy = l2_logistic_regression.score(logistic_testX, logistic_testY)
print(f"Logistic Regression Train Accuracy: {l2_logistic_train_accuracy.__format__('.4f')}")
print(f"Logistic Regression Test Accuracy: {l2_logistic_test_accuracy.__format__('.4f')}")

#Task 5
Naive Bayses Algorithm

#GaussianNB

In [ ]:
selected_features = ['num__Age', 'num__Future Potential', 'num__Total_Stats Score']

X_train_gnb = trainX[selected_features]
X_test_gnb = testX[selected_features]
y_train = trainX['Rating_Tier']
y_test = testX['Rating_Tier']

gnb = GaussianNB()
gnb.fit(X_train_gnb, y_train)

y_pred_gnb = gnb.predict(X_test_gnb)

print("Naïve Bayes Accuracy (3 Numerical Features):", accuracy_score(y_test, y_pred_gnb))
print(classification_report(y_test, y_pred_gnb))
cm = confusion_matrix(y_test,y_pred_gnb )
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Low', 'Mid', 'High', 'Elite'])
disp.plot(cmap='Blues')
plt.show()


#GaussianNB Without Scaling

In [ ]:
selected_features = ['Age', 'Future Potential', 'Total_Stats Score']

X_train_gnb = trainX_before_pipeline[selected_features]
X_test_gnb = testX_before_pipeline[selected_features]
y_train = trainX['Rating_Tier']
y_test = testX['Rating_Tier']

gnb = GaussianNB()
gnb.fit(X_train_gnb, y_train)

y_pred_gnb = gnb.predict(X_test_gnb)

print("Naïve Bayes Accuracy (3 Numerical Features):", accuracy_score(y_test, y_pred_gnb))
print(classification_report(y_test, y_pred_gnb))
cm = confusion_matrix(y_test,y_pred_gnb )
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Low', 'Mid', 'High', 'Elite'])
disp.plot(cmap='Blues')
plt.show()


No Change Occures because Naive Bayes is based on probability distributions rather than geometric distances. It calculates the conditional probability of each feature independently

#BernoulliNB

In [ ]:
columns_to_use = ['Position','Country','Team']
encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
X_train_class = encoder.fit_transform(trainX_before_pipeline[columns_to_use])
X_test_class = encoder.transform(testX_before_pipeline[columns_to_use])


In [ ]:

bnb = BernoulliNB()
bnb.fit(X_train_class, y_train)

y_train_pred_bnb = bnb.predict(X_train_class)
y_test_pred_bnb = bnb.predict(X_test_class)

print("Bernoulli NB Train Accuracy:", accuracy_score(y_train, y_train_pred_bnb))
print(classification_report(y_train, y_train_pred_bnb))

print("Bernoulli NB Test Accuracy:", accuracy_score(y_test, y_test_pred_bnb))
print(classification_report(y_test, y_test_pred_bnb))

cm = confusion_matrix(y_test, y_test_pred_bnb)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Low', 'Mid', 'High', 'Elite'])
disp.plot(cmap='Blues')
plt.show()

#Complement Naive Bayes

In [ ]:
minMaxScaler = MinMaxScaler()
X_train_non_negative = minMaxScaler.fit_transform(trainX_before_pipeline[numerical_features])
X_test_non_negative = minMaxScaler.transform(testX_before_pipeline[numerical_features])

X_train_full = np.concatenate((X_train_non_negative, X_train_class), axis=1)
X_test_full = np.concatenate((X_test_non_negative, X_test_class), axis=1)

X_train_full = pd.DataFrame(X_train_full)
X_test_full = pd.DataFrame(X_test_full)

In [ ]:
y_train = trainX['Rating_Tier']
y_test = testX['Rating_Tier']

cnb = ComplementNB()
cnb.fit(X_train_full, y_train)

y_pred_cnb = cnb.predict(X_test_full)

print("Complement Naïve Bayes Accuracy:", accuracy_score(y_test, y_pred_cnb))
print(classification_report(y_test, y_pred_cnb))
cm = confusion_matrix(y_test,y_pred_cnb )
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Low', 'Mid', 'High', 'Elite'])
disp.plot(cmap='Blues')
plt.show()

#Three model evaluation

In [ ]:
def full_evaluation(model, X_test, y_test, model_name):
    y_pred = model.predict(X_test)
    print(f"\n--- {model_name} Evaluation ---")
    print(classification_report(y_test, y_pred))

    plt.figure(figsize=(6, 5))
    cm = confusion_matrix(y_test, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Low', 'Mid', 'High', 'Elite'])
    disp.plot(cmap='Blues', ax=plt.gca())
    plt.title(f'Confusion Matrix - {model_name}')
    plt.show()

full_evaluation(gnb, X_test_gnb, y_test, "Gaussian Naive Bayes")
full_evaluation(bnb, X_test_class, y_test, "Bernoulli Naive Bayes")
full_evaluation(cnb, X_test_full, y_test, "Complement Naive Bayes")

# K-Fold

In [ ]:
best_reg_model = Pipeline([
    ('preprocessor', preprocessor),
    ('poly', PolynomialFeatures(degree=4)),
    ('ridge', Ridge(alpha=best_alpha_ridge))
])

In [ ]:
X = df.drop(columns=['Value Per M$'])
y = df['Value Per M$'].apply(lambda x: np.log(x) if x > 0 else 0)
kf =  KFold(n_splits=5, shuffle=True, random_state=42)

scores = cross_val_score(
    best_reg_model ,
    X,
    y,
    scoring='neg_mean_squared_error',
    cv = kf
)

rmse_scores = np.sqrt(-scores)

In [ ]:
print("RMSE for each fold:", rmse_scores)
print("Mean RMSE:", rmse_scores.mean())
print("STD:", rmse_scores.std())

In [ ]:
plt.bar(range(1,6), rmse_scores)
plt.axhline(y=rmse_scores.mean(), linestyle='--')

plt.xlabel("Fold")
plt.ylabel("RMSE")
plt.title("5-Fold Cross Validation (Ridge)")

plt.show()

#Stratified K-Fold

In [ ]:
numerical_features_for_classification = df.drop(columns=['Value Per M$', 'Overall_Rating']).select_dtypes(include=np.number).columns

preprocessor_classification = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_features_for_classification),
        ('cat', TargetEncoder(), categorical_features)
    ])

best_logistic_model = Pipeline([
    ('preprocessor', preprocessor_classification),
    ('lasso', LogisticRegression(penalty='l2', solver='saga', C=best_c,max_iter=10000))
])

In [ ]:
Q1 = df['Overall_Rating'].quantile(0.75)
Q2 = df['Overall_Rating'].quantile(0.82)
Q3 = df['Overall_Rating'].quantile(0.87)

X = df.drop(columns=['Overall_Rating'])
y = df['Overall_Rating'].apply(lambda x: categorize_rating(x, Q1, Q2, Q3))

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

LRscores = cross_val_score(
    best_logistic_model ,
    X,
    y,
    scoring='accuracy',
    cv = skf
)


In [ ]:
print("Accuracy for each fold:", LRscores)
print("Mean Accuracy:", LRscores.mean())
print("STD:", LRscores.std())

In [ ]:
plt.bar(range(1,6), LRscores)
plt.axhline(y=LRscores.mean(), linestyle='--')

plt.xlabel("Fold")
plt.ylabel("Accuracy")
plt.title("5-Fold Cross Validation (Lasso)")

plt.show()

#Naive Bayes K-Fold

In [ ]:
X = df[selected_features]

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

best_naive_bayes_model = GaussianNB()

NBscores = cross_val_score(
    best_naive_bayes_model ,
    X,
    y,
    scoring='accuracy',
    cv = skf
)


In [ ]:
print("Accuracy for each fold:", NBscores)
print("Mean Accuracy:", NBscores.mean())
print("STD:", NBscores.std())

In [ ]:
plt.bar(range(1,6), NBscores)
plt.axhline(y=NBscores.mean(), linestyle='--')

plt.xlabel("Fold")
plt.ylabel("Accuracy")
plt.title("5-Fold Cross Validation (GaussianNB)")

plt.show()

In [ ]:
fig = go.Figure(data=[
    go.Bar(name='Naive Bayes Scores', x=list(range(1,6)),y=NBscores),
    go.Bar(name='Logistic Regression Scores', x=list(range(1,6)),y=LRscores),
])
fig.update_layout(barmode='group')
fig.show()

#Model Comparison

*Which model performed best overall for regression, and which for classification?* justify briefly

Best Regression Model: `Ridge Polynomial Regression`

The Ridge Polynomial Regression achieved the lowest RMSE (0.2387) on the test set among the regression models, indicating better prediction accuracy compared to normal linear regression (RMSE: 0.3114) and Lasso Polynomial Regression (RMSE: 0.2437).


Best Classification Model: `Lasso Logistic Regression`

The Lasso Logistic Regression (L1 penalized) demonstrated the highest test accuracy of approximately 0.8922. This is superior to the other Naive Bayes models, which had accuracies around 0.70 (GaussianNB) and 0.52 (BernoulliNB and ComplementNB).

*Is classification easier or harder than regression on this dataset? Why?*

`Classification` was harder as there was no clear target and we had to create the classification target

#Regularization Analysis

*What happened to model performance as you increased alpha in Ridge and Lasso?*

`Ridge`: There wasn't much difference on huge differences in alpha values.


`Lasso`: Large values for alpha made the model eliminate all the features and decrease their coefficients to 0.

*Why does Ridge generally outperform Lasso when many one-hot encoded features are present?*

When you have one-hot encoded features representing a single categorical variable, they are inherently related. If Lasso selects one of these features, it tends to arbitrarily pick only one or a few from the group and shrink the coefficients of the others to zero. Ridge, on the other hand, tends to shrink the coefficients of correlated features towards each other, effectively performing a more 'grouped' selection. This means it's more likely to keep all the relevant dummy variables together, which often leads to better predictive performance when the true underlying relationship involves the entire categorical variable.

---

# **ML Assignment 3**

---



# Advanced Classification Models



*   Instance-Based **"KNN"**




In [ ]:
trainX_before_pipeline.drop(columns=['Overall_Rating'], inplace=True)
testX_before_pipeline.drop(columns=['Overall_Rating'], inplace=True)

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC,SVR
from sklearn.model_selection import GridSearchCV, KFold,StratifiedKFold
from sklearn.preprocessing import LabelEncoder

knn = KNeighborsClassifier()

knn_pipeline = Pipeline(
    steps=[
        ('preprocessor', preprocessor_classification),
        ('knn', knn)
    ]
)
param_grid = {
    'knn__n_neighbors': range(1,30,2),
    'knn__metric' : ['minkowski','euclidean','manhattan']
    }
stf_kf =  StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
grid_search = GridSearchCV(knn_pipeline, param_grid, cv=stf_kf, scoring='accuracy')

grid_search.fit(trainX_before_pipeline, y_train)

y_pred = grid_search.predict(testX_before_pipeline)

print(accuracy_score(y_test, y_pred))



In [ ]:
print(grid_search.best_params_)
print(grid_search.best_score_)



*   Kernel-Based **"SVC"**




In [ ]:
best_knn_model = KNeighborsClassifier(metric=grid_search.best_params_['knn__metric'], n_neighbors=grid_search.best_params_['knn__n_neighbors'])

In [ ]:
svc_models = {
    "SVC (RBF kernel)": SVC(kernel='rbf'),
    "Linear SVM": SVC(kernel='linear'),
    "Polynomial SVM": SVC(kernel='poly', degree=4)
}
for name, model in svc_models.items():
    full_pipeline = Pipeline(steps=[
        ('preprocessor', preprocessor_classification),
        ('classifier', model)
    ])

    # Fit the pipeline (cleans and trains in one go)
    full_pipeline.fit(trainX_before_pipeline, y_train)

    # Predict using raw X_test (pipeline handles the scaling/encoding)
    y_pred = full_pipeline.predict(testX_before_pipeline)

    print(f"\n{name} Results:")
    print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
    print(classification_report(y_test, y_pred))

In [ ]:
best_svc_model = SVC(kernel='rbf')

svc_pipeline = Pipeline(
    steps=[
        ('preprocessor', preprocessor_classification),
        ('classifier', best_svc_model)
    ]
)

param_grid = {
    'classifier__C': [0.1,1,10],
    'classifier__gamma': [0.1,1,10],
}

stf_kf =  StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

grid_search = GridSearchCV(svc_pipeline, param_grid, cv=stf_kf, scoring='accuracy')


grid_search.fit(trainX_before_pipeline, y_train)

y_pred = grid_search.predict(testX_before_pipeline)

print(grid_search.best_params_)
print(grid_search.best_score_)

print("Test Accuracy: ",accuracy_score(y_test, y_pred))

sns.heatmap(confusion_matrix(y_test, y_pred), annot=True, fmt='d')
plt.show


In [ ]:
best_svc_model = SVC(kernel='rbf',probability=True,C=grid_search.best_params_['classifier__C'],gamma=grid_search.best_params_['classifier__gamma'])



*   Tree-Based **"Random Forest"**



In [ ]:
from sklearn.ensemble import RandomForestClassifier

In [ ]:
rf_clf = RandomForestClassifier(
max_features = 'sqrt',
min_samples_leaf = 2,
random_state = 42,
n_jobs = -1
)

rf_pipeline = Pipeline(
    steps=[
        ('preprocessor', preprocessor_classification),
        ('classifier', rf_clf)
    ]
)
param_grid = {
    'classifier__n_estimators': [100, 200, 300],
    'classifier__max_depth': [None, 10,20]
}

grid_search = GridSearchCV(rf_pipeline, param_grid, cv=stf_kf, scoring='accuracy')

grid_search.fit(trainX_before_pipeline, y_train)

y_pred = grid_search.predict(testX_before_pipeline)

print(grid_search.best_params_)
print(grid_search.best_score_)
print("Test Accuracy: ",accuracy_score(y_test, y_pred))
sns.heatmap(confusion_matrix(y_test, y_pred), annot=True, fmt='d')
plt.show


In [ ]:
best_random_forest = RandomForestClassifier(
max_depth = grid_search.best_params_['classifier__max_depth'],
n_estimators = grid_search.best_params_['classifier__n_estimators'],
max_features = 'sqrt',
min_samples_leaf = 2,
random_state = 42,
n_jobs = -1)

In [ ]:
from sklearn.tree import DecisionTreeClassifier

In [ ]:
dt_clf = DecisionTreeClassifier()

dt_pipeline = Pipeline(
    steps=[
        ('preprocessor', preprocessor_classification),
        ('classifier', dt_clf)
    ]
)
param_grid = {
    'classifier__max_depth': range(1,5)
}

grid_search = GridSearchCV(dt_pipeline, param_grid, cv=stf_kf, scoring='accuracy')

grid_search.fit(trainX_before_pipeline, y_train)

y_pred = grid_search.predict(testX_before_pipeline)

print(grid_search.best_params_)
print(grid_search.best_score_)
print("Test Accuracy: ",accuracy_score(y_test, y_pred))
sns.heatmap(confusion_matrix(y_test, y_pred), annot=True, fmt='d')
plt.show


In [ ]:
models = {
'Logistic Regression' : best_logistic_model.named_steps['lasso'],
'Random Forest': best_random_forest,
'K-Nearest Neighbors': best_knn_model,
'SVM': best_svc_model,
'Decision Tree': DecisionTreeClassifier(max_depth=4)
}

In [ ]:
for name, model in models.items():
  pipeline = Pipeline(
    steps=[
        ('preprocessor', preprocessor_classification),
        ('classifier', model)
    ]
  )
  pipeline.fit(trainX_before_pipeline, y_train)
  y_pred = pipeline.predict(testX_before_pipeline)
  # Evaluate
  acc = accuracy_score(y_test, y_pred)
  print(f"{name:20s} : {acc:.4f}")

In [ ]:
from sklearn.ensemble import VotingClassifier

In [ ]:
print("\n" + "=" * 60)
print("HARD VOTING CLASSIFIER")
print("=" * 60)
# Create hard voting ensemble
voting_hard = VotingClassifier(
estimators=[
('rf', best_random_forest),
('knn', best_knn_model),
('svm', best_svc_model),
],
voting='hard' # Majority vote
)

In [ ]:
voting_hard_pipeline = Pipeline(
    steps=[
        ('preprocessor', preprocessor_classification),
        ('classifier', voting_hard)
    ]
)

voting_hard_pipeline.fit(trainX_before_pipeline, y_train)
y_pred_hard = voting_hard_pipeline.predict(testX_before_pipeline)
hard_acc = accuracy_score(y_test, y_pred_hard)
print(f"Hard Voting Accuracy: {hard_acc:.4f}")

In [ ]:
voting_soft = VotingClassifier(
estimators=[
('rf', best_random_forest),
('knn', best_knn_model),
('svm', best_svc_model),
],
voting='soft' # Average probabilities
)
# Train on scaled data

voting_soft_pipeline = Pipeline(
    steps=[
        ('preprocessor', preprocessor_classification),
        ('classifier', voting_soft)
    ]
)
voting_soft_pipeline.fit(trainX_before_pipeline, y_train)
y_pred_soft = voting_soft_pipeline.predict(testX_before_pipeline)
soft_acc = accuracy_score(y_test, y_pred_soft)
print(f"Soft Voting Accuracy: {soft_acc:.4f}")

In [ ]:
Q1 = df['Overall_Rating'].quantile(0.75)
Q2 = df['Overall_Rating'].quantile(0.82)
Q3 = df['Overall_Rating'].quantile(0.95)

X = df.drop(columns=['Overall_Rating'])
y = df['Overall_Rating'].apply(lambda x: categorize_rating(x, Q1, Q2, Q3))
stf_kf =  StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

scores = cross_val_score(
    voting_hard_pipeline ,
    X,
    y,
    scoring='accuracy',
    cv = stf_kf
)

In [ ]:
print("Scores for each fold:", scores)
print("Mean Scores:", scores.mean())
print("STD:", scores.std())

In [ ]:
plt.bar(range(1,6), scores)
plt.axhline(y=scores.mean(), linestyle='--')

plt.xlabel("Fold")
plt.ylabel("Accuracy")
plt.title("5-Fold Cross Validation (Hard)")

plt.show()

# Advanced Regression Models



*   Instance-Based **"KNN-Regressor"**




In [ ]:
reg_Train_X , reg_Test_X ,reg_Train_Y, reg_Test_Y = train_test_split(df.drop(columns=['Value Per M$']), df['Value Per M$'], test_size=0.2, random_state=42)

In [ ]:
reg_Train_Y = reg_Train_Y.apply(lambda x: log(x) if x > 0 else 0)
reg_Test_Y = reg_Test_Y.apply(lambda x: log(x) if x > 0 else 0)

In [ ]:
from sklearn.neighbors import KNeighborsRegressor
from sklearn.model_selection import GridSearchCV, KFold,StratifiedKFold
from sklearn.preprocessing import LabelEncoder

knn = KNeighborsRegressor()

knn_pipeline = Pipeline(
    steps=[
        ('preprocessor', preprocessor),
        ('knn', knn)
    ]
)
param_grid = {
    'knn__n_neighbors': range(1,30,2),
    'knn__metric' : ['minkowski','euclidean','manhattan']
    }
kf =  KFold(n_splits=5, shuffle=True, random_state=42)
grid_search = GridSearchCV(knn_pipeline, param_grid, cv=kf, scoring='neg_mean_squared_error')

grid_search.fit(reg_Train_X, reg_Train_Y)

y_pred = grid_search.predict(reg_Test_X)

print(f"R2 Score: {r2_score(reg_Test_Y, y_pred):.4f}")


In [ ]:
print(grid_search.best_params_)
print(grid_search.best_score_*-1)

In [ ]:
best_knn_model = KNeighborsRegressor(metric=grid_search.best_params_['knn__metric'], n_neighbors=grid_search.best_params_['knn__n_neighbors'])



*   Kernel-Based **"SVR"**




In [ ]:
svr_models = {
    "SVR (RBF kernel)": SVR(kernel='rbf'),
    "Linear SVM": SVR(kernel='linear'),
    "Polynomial SVM": SVR(kernel='poly', degree=4)
}
for name, model in svr_models.items():
    full_pipeline = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('regressor', model)
    ])

    # Fit the pipeline (cleans and trains in one go)
    full_pipeline.fit(reg_Train_X, reg_Train_Y)

    # Predict using raw X_test (pipeline handles the scaling/encoding)
    y_pred = full_pipeline.predict(reg_Test_X)

    print(f"\n{name} Results:")
    print(f"R2 Score: {r2_score(reg_Test_Y, y_pred):.4f}")

In [ ]:
best_svr_model = SVR(kernel='rbf')

svr_pipeline = Pipeline(
    steps=[
        ('preprocessor', preprocessor),
        ('regressor', best_svr_model)
    ]
)

param_grid = {
    'regressor__C': [0.1,1,10],
    'regressor__gamma': [0.1,1,10],
}

kf =  KFold(n_splits=5, shuffle=True, random_state=42)

grid_search = GridSearchCV(svr_pipeline, param_grid, cv=kf, scoring='neg_mean_squared_error')


grid_search.fit(reg_Train_X, reg_Train_Y)

y_pred = grid_search.predict(reg_Test_X)

print(grid_search.best_params_)
print(grid_search.best_score_ * -1)

print(f"Test R2 Score: {r2_score(reg_Test_Y, y_pred):.4f}")


In [ ]:
best_svr_model = SVR(kernel='rbf',C=grid_search.best_params_['regressor__C'],gamma=grid_search.best_params_['regressor__gamma'])



*   Tree-Based **"Random Forest"**



In [ ]:
from sklearn.ensemble import RandomForestRegressor

In [ ]:
rf_reg = RandomForestRegressor(
max_features = 'sqrt', # sqrt(n_features) candidates per split
min_samples_leaf = 2,
random_state = 42,
n_jobs = -1
)

rf_pipeline = Pipeline(
    steps=[
        ('preprocessor', preprocessor),
        ('regressor', rf_reg)
    ]
)
param_grid = {
    'regressor__n_estimators': [100, 200, 300],
    'regressor__max_depth': [None, 10,20]
}

grid_search = GridSearchCV(rf_pipeline, param_grid, cv=kf, scoring='neg_mean_squared_error')

grid_search.fit(reg_Train_X, reg_Train_Y)

y_pred = grid_search.predict(reg_Test_X)

print(grid_search.best_params_)
print(grid_search.best_score_ * -1)
print(f"Test R2 Score: {r2_score(reg_Test_Y, y_pred):.4f}")


In [ ]:
best_random_forest = RandomForestRegressor(
max_depth = grid_search.best_params_['regressor__max_depth'],
n_estimators = grid_search.best_params_['regressor__n_estimators'],
max_features = 'sqrt', # sqrt(n_features) candidates per split
min_samples_leaf = 2,
random_state = 42,
n_jobs = -1)

In [ ]:
models = {
'Random Forest': best_random_forest,
'K-Nearest Neighbors': best_knn_model,
'SVM': best_svr_model,
}

In [ ]:
from sklearn.metrics import r2_score

In [ ]:
for name, model in models.items():
  pipeline = Pipeline(
    steps=[
        ('preprocessor', preprocessor),
        ('regressor', model)
    ]
  )
  pipeline.fit(reg_Train_X, reg_Train_Y)
  y_pred = pipeline.predict(reg_Test_X)
  # Evaluate
  current_r2_score = r2_score(reg_Test_Y, y_pred)
  print(f"{name:20s} : {current_r2_score:.4f}")

In [ ]:
from sklearn.ensemble import VotingRegressor

In [ ]:
print("\n" + "=" * 60)
print("VOTING REGRESSOR")
print("=" * 60)
# Create voting ensemble
voting = VotingRegressor(
estimators=[
('rf', best_random_forest),
('knn', best_knn_model),
('svm', best_svr_model),
])

In [ ]:
voting_pipeline = Pipeline(
    steps=[
        ('preprocessor', preprocessor),
        ('regressor', voting)
    ]
)

voting_pipeline.fit(reg_Train_X, reg_Train_Y)
y_pred_hard = voting_pipeline.predict(reg_Test_X)
current_r2_score = r2_score(reg_Test_Y, y_pred_hard)
print(f"Voting R2 Score: {current_r2_score:.4f}")

In [ ]:
scores = cross_val_score(
    voting_pipeline ,
    df.drop(columns=['Value Per M$']),
    df['Value Per M$'],
    scoring='neg_mean_squared_error',
    cv = kf
)

rmse_scores = np.sqrt(-scores)

In [ ]:
print("RMSE for each fold:", rmse_scores)
print("Mean RMSE:", rmse_scores.mean())
print("STD:", rmse_scores.std())

In [ ]:
plt.bar(range(1,6), rmse_scores)
plt.axhline(y=rmse_scores.mean(), linestyle='--')

plt.xlabel("Fold")
plt.ylabel("RMSE")
plt.title("5-Fold Cross Validation (Voting)")

plt.show()

In [ ]:
def predict(data):

  def return_from_log(number):
    return math.e**number

  category_map = {
      0 : 'Low',
      1 : 'Medium',
      2 : 'High',
      3 : 'Elite'
  }

  predicted_price = voting_pipeline.predict(data)
  predicted_price = return_from_log(predicted_price[0])
  predicted_tier = voting_soft_pipeline.predict(data)
  predicted_tier = category_map[predicted_tier[0]]

  print(f"The Predicted Price is: {predicted_price}")
  print(f"The Predicted Tier is: {predicted_tier}")


row = df.sample(1)
print(row)
data = row.drop(columns=['Value Per M$'])
predict(data)